In [1]:
import os, time
from datetime import datetime, timedelta, timezone
import pandas as pd
import requests

HEADERS = {"User-Agent": "african-wikipedia-live-activity/0.1 (https://github.com/Chichay317/african-wikipedia-live-activity)"}
CANDIDATES = {
    "af": "Afrikaans", "ak": "Akan", "am": "Amharic", "ann": "Obolo", "bm": "Bambara", "dag": "Dagbani", "din": "Dinka",
    "ee": "Ewe", "fat": "Fante", "ff": "Fula", "fon": "Fon", "gaa": "Ga", "guw": "Gun", "ha": "Hausa", "ig": "Igbo",
    "igl": "Igala", "kab": "Kabyle", "kbp": "Kabiye", "kcg": "Tyap", "kg": "Kongo", "ki": "Kikuyu", "kus": "Kusaal",
    "lg": "Luganda", "ln": "Lingala", "mg": "Malagasy", "mos": "Moore", "nr": "Southern Ndebele", "nso": "Northern Sotho",
    "ny": "Chichewa", "om": "Oromo", "rn": "Kirundi", "rw": "Kinyarwanda", "sg": "Sango", "shi": "Tachelhit",
    "sn": "Shona", "so": "Somali", "ss": "Swati", "st": "Sesotho", "sw": "Swahili", "ti": "Tigrinya", "tn": "Setswana",
    "ts": "Tsonga", "tum": "Tumbuka", "tw": "Twi", "ve": "Venda", "wo": "Wolof", "xh": "Xhosa", "yo": "Yoruba",
    "zgh": "Standard Moroccan Tamazight", "zu": "Zulu",
}

r = requests.get("https://meta.wikimedia.org/w/api.php", headers=HEADERS, timeout=60,
                 params={"action": "sitematrix", "smtype": "language", "format": "json", "formatversion": 2})
r.raise_for_status()
rows = []
for lang in r.json()["sitematrix"].values():
    if not isinstance(lang, dict) or lang.get("code") not in CANDIDATES:
        continue
    for site in lang.get("site", []):
        if site.get("code") == "wiki":                               
            rows.append({"code": lang["code"], "language": CANDIDATES[lang["code"]], "wiki": site["dbname"],
                         "url": site["url"], "closed": bool(site.get("closed", False))})
wikis = pd.DataFrame(rows).sort_values("code").reset_index(drop=True)
print(f"{len(wikis)} of {len(CANDIDATES)} candidate languages have a Wikipedia; "
      f"{(~wikis.closed).sum()} are open, {wikis.closed.sum()} closed")
print("No Wikipedia found for:", sorted(set(CANDIDATES) - set(wikis.code)) or "none")
print("Closed:", wikis[wikis.closed].language.tolist() or "none")
os.makedirs("../data", exist_ok=True)
wikis.to_csv("../data/african_wikis.csv", index=False)

49 of 50 candidate languages have a Wikipedia; 48 are open, 1 closed
No Wikipedia found for: ['gaa']
Closed: ['Akan']


In [2]:
def recent_changes(url, start, end):
    params = {"action": "query", "list": "recentchanges", "rcstart": f"{end:%Y-%m-%dT%H:%M:%SZ}",
              "rcend": f"{start:%Y-%m-%dT%H:%M:%SZ}", "rcprop": "ids|timestamp|flags|user|title",
              "rclimit": 500, "format": "json", "formatversion": 2}
    changes = []
    while True:
        r = requests.get(url + "/w/api.php", params=params, headers=HEADERS, timeout=60)
        r.raise_for_status()
        data = r.json()
        changes += data["query"]["recentchanges"]
        time.sleep(0.2)                                     
        if "continue" not in data:
            return changes
        params.update(data["continue"])

end = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
start = end - timedelta(hours=24)
summary = []
for w in wikis[~wikis.closed].itertuples():
    rc = pd.DataFrame(recent_changes(w.url, start, end))
    summary.append({"language": w.language, "wiki": w.wiki, "changes": len(rc),
                    "edits": int((rc.type == "edit").sum()) if len(rc) else 0,
                    "new_pages": int((rc.type == "new").sum()) if len(rc) else 0,
                    "by_bots": int(rc.get("bot", pd.Series(dtype=bool)).fillna(False).sum()) if len(rc) else 0,
                    "users": rc.user.nunique() if len(rc) else 0})
summary = pd.DataFrame(summary).sort_values("changes", ascending=False).reset_index(drop=True)
summary["bot_share"] = (summary.by_bots / summary.changes.where(summary.changes > 0)).round(2)
print(f"Recent changes, {start:%Y-%m-%d %H:00} to {end:%Y-%m-%d %H:00} UTC:")
print(summary.to_string())
total = summary.changes.sum()
print(f"\nAll {len(summary)} wikis together: {total:,} changes in 24 hours, about {total / 1440:.1f} per minute")
print("Wikis with no changes at all:", summary[summary.changes == 0].language.tolist() or "none")
summary.to_csv("../data/rc_24h_summary.csv", index=False)

Recent changes, 2026-10-12 00:00 to 2026-10-13 00:00 UTC:
                       language     wiki  changes  edits  new_pages  by_bots  users  bot_share
0                       Swahili   swwiki      951    128        149      149     24       0.16
1                         Hausa   hawiki      619    250         80      275     35       0.44
2                     Afrikaans   afwiki      515    246        168      215     29       0.42
3                          Fula   ffwiki      331    184         47        8      4       0.02
4                         Igala  iglwiki       76     28          5        0      3       0.00
5                   Kinyarwanda   rwwiki       75     53          0       39      8       0.52
6                       Luganda   lgwiki       68     42          6        1      7       0.01
7                          Igbo   igwiki       67     50          3       58      8       0.87
8                      Malagasy   mgwiki       37     23          9        0      8    

In [3]:
import json

STREAM = "https://stream.wikimedia.org/v2/stream/recentchange"
african = set(wikis.loc[~wikis.closed, "wiki"])

def read_stream(seconds):
    kept, total, canary, nbytes, t0 = [], 0, 0, 0, time.time()
    with requests.get(STREAM, headers=HEADERS, stream=True, timeout=(10, 60)) as r:
        r.raise_for_status()
        for raw in r.iter_lines():
            nbytes += len(raw) + 1
            line = raw.decode("utf-8", errors="replace")
            if line.startswith("data: "):     
                total += 1
                try:
                    e = json.loads(line[6:])
                except json.JSONDecodeError:
                    continue
                if e.get("meta", {}).get("domain") == "canary": 
                    canary += 1
                elif e.get("wiki") in african:
                    e["received_at"] = time.time()
                    kept.append(e)
            if time.time() - t0 > seconds:
                break
    return kept, total, canary, nbytes, time.time() - t0

window_start = datetime.now(timezone.utc)
events, total, canary, nbytes, secs = read_stream(180)
window_end = datetime.now(timezone.utc)

print(f"Read the whole feed for {secs:.0f} s: {total:,} events ({total / secs:.0f} per second), "
      f"{nbytes / 1e6:.1f} MB, which is about {nbytes / secs * 3600 / 1e9:.2f} GB per hour")
print(f"Canary (test) events skipped: {canary}")
print(f"Kept from African-language Wikipedias: {len(events)} ({100 * len(events) / max(total, 1):.3f}% of the feed)")
if events:
    ev = pd.DataFrame(events)
    ev["lag_s"] = ev.received_at - ev.timestamp
    print("\nBy wiki and type:")
    print(ev.groupby(["wiki", "type"]).size().to_string())
    print(f"\nSeconds from the change being made to it reaching us: median {ev.lag_s.median():.1f}, max {ev.lag_s.max():.1f}")
    sample = {k: events[0].get(k) for k in ["wiki", "type", "id", "title", "user", "bot", "timestamp", "length", "revision"]}
    sample["meta"] = {k: events[0]["meta"].get(k) for k in ["id", "dt", "topic", "partition", "offset"]}
    print("\nOne event, selected fields:")
    print(json.dumps(sample, indent=2, ensure_ascii=False))

Read the whole feed for 180 s: 4,176 events (23 per second), 6.4 MB, which is about 0.13 GB per hour
Canary (test) events skipped: 0
Kept from African-language Wikipedias: 1 (0.024% of the feed)

By wiki and type:
wiki    type
swwiki  edit    1

Seconds from the change being made to it reaching us: median 0.1, max 0.1

One event, selected fields:
{
  "wiki": "swwiki",
  "type": "edit",
  "id": 5173384,
  "title": "Wikipedia:Mradi wa Nchi",
  "user": "Gayle-Bot",
  "bot": true,
  "timestamp": 1791851063,
  "length": {
    "old": 21885,
    "new": 21884
  },
  "revision": {
    "old": 1607145,
    "new": 1607214
  },
  "meta": {
    "id": "110b7986-d006-4bff-aa8f-5d7eed577578",
    "dt": "2026-10-13T00:24:24.294Z",
    "topic": "eqiad.mediawiki.recentchange",
    "partition": 0,
    "offset": 6605469116
  }
}


In [4]:
check_from = window_start.replace(microsecond=0) + timedelta(seconds=10)     
check_to = window_end.replace(microsecond=0) - timedelta(seconds=60)     
in_window = lambda ts: check_from.timestamp() <= ts < check_to.timestamp()
stream_ids = {(e["wiki"], e["id"]) for e in events if in_window(e["timestamp"]) and e.get("id") is not None}

api_ids = set()
for w in wikis[~wikis.closed].itertuples():
    for c in recent_changes(w.url, check_from, check_to):
        ts = datetime.fromisoformat(c["timestamp"].replace("Z", "+00:00")).timestamp()
        if in_window(ts):
            api_ids.add((w.wiki, c["rcid"]))

print(f"Window checked: {check_from:%H:%M:%S} to {check_to:%H:%M:%S} UTC")
print(f"Changes in the API (Wikipedia's own record): {len(api_ids)}")
print(f"Changes we caught from the feed:            {len(stream_ids)}")
print(f"In both: {len(api_ids & stream_ids)} | only in API (we missed): {len(api_ids - stream_ids)} "
      f"| only in feed: {len(stream_ids - api_ids)}")
for label, ids in [("Missed", api_ids - stream_ids), ("Feed only", stream_ids - api_ids)]:
    if ids:
        print(f"{label}:", sorted(ids)[:10])

Window checked: 00:24:19 to 00:26:09 UTC
Changes in the API (Wikipedia's own record): 1
Changes we caught from the feed:            1
In both: 1 | only in API (we missed): 0 | only in feed: 0


In [5]:
def oldest_change(url):
    r = requests.get(url + "/w/api.php", headers=HEADERS, timeout=60,
                     params={"action": "query", "list": "recentchanges", "rcdir": "newer", "rclimit": 1,
                             "rcprop": "timestamp", "format": "json", "formatversion": 2})
    r.raise_for_status()
    rc = r.json()["query"]["recentchanges"]
    return datetime.fromisoformat(rc[0]["timestamp"].replace("Z", "+00:00")) if rc else None

now = datetime.now(timezone.utc)
memory = []
for w in wikis[wikis.wiki.isin(summary.wiki.head(8))].itertuples():   
    oldest = oldest_change(w.url)
    memory.append({"language": w.language, "oldest_change_listed": oldest,
                   "days_back": round((now - oldest).total_seconds() / 86400, 1) if oldest else None})
    time.sleep(0.2)
print("How far back each Wikipedia's API remembers changes:")
print(pd.DataFrame(memory).to_string(index=False))

How far back each Wikipedia's API remembers changes:
   language      oldest_change_listed  days_back
  Afrikaans 2026-09-13 02:28:27+00:00       29.9
       Fula 2026-09-13 02:26:16+00:00       29.9
      Hausa 2026-09-13 01:30:52+00:00       30.0
       Igbo 2026-09-13 01:08:56+00:00       30.0
      Igala 2026-09-13 07:40:32+00:00       29.7
    Luganda 2026-09-13 03:55:31+00:00       29.9
Kinyarwanda 2026-09-13 06:01:37+00:00       29.8
    Swahili 2026-09-13 00:29:29+00:00       30.0


In [6]:
def none_if_zero(x):
    return None if x in (0, None) else x

def lengths(kind, old, new):
    if kind not in ("edit", "new"):
        return None, None
    return (None if kind == "new" else old), new

def from_feed(e):
    len_old, len_new = lengths(e["type"], (e.get("length") or {}).get("old"), (e.get("length") or {}).get("new"))
    return {"wiki": e["wiki"], "rc_id": e["id"], "type": e["type"], "namespace": e.get("namespace"),
            "title": e.get("title"), "user": e.get("user"), "bot": bool(e.get("bot")),
            "changed_at": datetime.fromtimestamp(e["timestamp"], timezone.utc).isoformat(),
            "rev_old": none_if_zero((e.get("revision") or {}).get("old")), "rev_new": none_if_zero((e.get("revision") or {}).get("new")),
            "len_old": len_old, "len_new": len_new}

def from_api(c, wiki):
    len_old, len_new = lengths(c["type"], c.get("oldlen"), c.get("newlen"))
    return {"wiki": wiki, "rc_id": c["rcid"], "type": c["type"], "namespace": c.get("ns"),
            "title": c.get("title"), "user": c.get("user"), "bot": bool(c.get("bot")),
            "changed_at": datetime.fromisoformat(c["timestamp"].replace("Z", "+00:00")).isoformat(),
            "rev_old": none_if_zero(c.get("old_revid")), "rev_new": none_if_zero(c.get("revid")),
            "len_old": len_old, "len_new": len_new}

e = events[0]
url = wikis.set_index("wiki").url[e["wiki"]]
t = datetime.fromtimestamp(e["timestamp"], timezone.utc)
r = requests.get(url + "/w/api.php", headers=HEADERS, timeout=60,
                 params={"action": "query", "list": "recentchanges", "rcstart": f"{t:%Y-%m-%dT%H:%M:%SZ}",
                         "rcend": f"{t:%Y-%m-%dT%H:%M:%SZ}", "rcprop": "ids|timestamp|flags|user|title|sizes",
                         "rclimit": 50, "format": "json", "formatversion": 2})
r.raise_for_status()
match = [c for c in r.json()["query"]["recentchanges"] if c["rcid"] == e["id"]]
feed_row, api_row = from_feed(e), (from_api(match[0], e["wiki"]) if match else None)
print(pd.DataFrame({"from the feed": feed_row, "from the API": api_row or {}}).to_string())
print("\nIdentical after conversion:", feed_row == api_row)

                        from the feed               from the API
wiki                           swwiki                     swwiki
rc_id                         5173384                    5173384
type                             edit                       edit
namespace                           4                          4
title         Wikipedia:Mradi wa Nchi    Wikipedia:Mradi wa Nchi
user                        Gayle-Bot                  Gayle-Bot
bot                              True                       True
changed_at  2026-10-13T00:24:23+00:00  2026-10-13T00:24:23+00:00
rev_old                       1607145                    1607145
rev_new                       1607214                    1607214
len_old                         21885                      21885
len_new                         21884                      21884

Identical after conversion: True
